# SoninkeGPT – OmniVoice : génération audio par lots de 20

Ce notebook lit le lexique de `aydiarra-star/soninkeGPT`, génère les WAV manquants avec OmniVoice et ta voix de référence, puis sauvegarde les fichiers et le manifeste dans Google Drive. Il ne publie rien automatiquement : tous les nouveaux audios restent **À ÉCOUTER / NON VALIDÉS**.

**Avant de lancer :** dans Google Drive, crée `SONINKE_TTS_LIBRE/references/` et place-y `daga_goli.wav` et `li_taxu.wav`. Les transcriptions utilisées sont exactement `Daga goli.` et `Li taxu.`. Choisis un environnement d'exécution Colab avec GPU (Runtime → Change runtime type → T4 GPU si proposé).

Les exécutions sont limitées à 20 mots. Les WAV existants ne sont pas remplacés et le manifeste permet de reprendre au prochain lancement.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess, shutil
DRIVE_ROOT = Path('/content/drive/MyDrive/SONINKE_TTS_LIBRE')
REF_DIR = DRIVE_ROOT / 'references'
OUT_DIR = DRIVE_ROOT / 'audio_a_valider'
REF_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
REF1 = REF_DIR / 'daga_goli.wav'
REF2 = REF_DIR / 'li_taxu.wav'

# Aider si les références sont dans un autre sous-dossier de Mon Drive ou portent une extension différente.
def locate_reference(keywords):
    exts = {'.wav', '.ogg', '.opus', '.mp3', '.m4a', '.flac', '.webm'}
    hits = []
    for p in Path('/content/drive/MyDrive').rglob('*'):
        if p.is_file() and p.suffix.lower() in exts and any(k in p.stem.lower().replace('-', '_').replace(' ', '_') for k in keywords):
            hits.append(p)
    return hits

def prepare_reference(target, keywords):
    if target.exists() and target.stat().st_size > 44:
        source = target
    else:
        hits = locate_reference(keywords)
        if not hits:
            raise FileNotFoundError(f"Référence introuvable pour {target.name}. Mets le fichier dans {REF_DIR} ; fichiers audio recherchés avec mots-clés {keywords}.")
        source = hits[0]
        print('Référence trouvée :', source)
    # Réencoder vers un vrai WAV PCM mono 24 kHz, même si l'extension d'origine est trompeuse.
    tmp = target.with_name(target.stem + '_converted.wav')
    cmd = ['ffmpeg', '-y', '-i', str(source), '-ac', '1', '-ar', '24000', '-c:a', 'pcm_s16le', str(tmp)]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0 or not tmp.exists() or tmp.stat().st_size <= 44:
        raise RuntimeError(f"Impossible de lire/convertir {source}. Vérifie que le fichier audio s'ouvre et contient bien la voix. Détail ffmpeg : {proc.stderr[-1200:]}")
    tmp.replace(target)
    print('Référence prête :', target)

print('Dossier de sortie :', OUT_DIR)
prepare_reference(REF1, ['daga_goli', 'daga', 'goli'])
prepare_reference(REF2, ['li_taxu', 'taxu'])


In [ ]:
# Installer OmniVoice depuis le dépôt officiel, puis vérifier dans CE noyau Colab.
import os, subprocess, sys, importlib.util
REPO = Path('/content/soninkeGPT')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/aydiarra-star/soninkeGPT.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'soundfile', 'pandas', 'numpy'], check=True)
# Source officielle : évite qu'un paquet absent ou mal installé laisse le notebook continuer.
install = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--upgrade', 'git+https://github.com/k2-fsa/OmniVoice.git'], capture_output=True, text=True)
if install.returncode != 0:
    print(install.stdout[-2000:])
    print(install.stderr[-6000:])
    raise RuntimeError('Installation OmniVoice échouée. Arrêt avant la génération audio.')
print('Installation terminée. Vérification dans le noyau Colab…')
try:
    from omnivoice import OmniVoice
    print('✅ Import OmniVoice réussi dans le noyau Colab.')
except Exception as exc:
    print('Python utilisé :', sys.executable)
    print('Module détecté :', importlib.util.find_spec('omnivoice'))
    raise RuntimeError(f'OmniVoice installé mais import impossible : {exc}') from exc
print('Dépôt du lexique et dépendances installés.')


In [ ]:
# Vérifier GPU, lexique et références avant toute génération
import csv, json, hashlib, re, shutil, time
import numpy as np
import pandas as pd
import soundfile as sf
import torch
assert torch.cuda.is_available(), 'ARRÊT : GPU CUDA indisponible. Dans Colab, active un accélérateur GPU puis relance.'
LEXICON = REPO / 'data/source/lexique-soninke-fr-base.csv'
assert LEXICON.exists(), f'Lexique introuvable : {LEXICON}'
df = pd.read_csv(LEXICON, sep=';', dtype=str, keep_default_na=False)
assert 'soninke' in df.columns, f'Colonne soninke absente. Colonnes : {list(df.columns)}'
df['soninke'] = df['soninke'].astype(str).str.strip()
words = sorted({w for w in df['soninke'] if w and w.lower() != 'nan'}, key=str.casefold)
assert words, 'Aucun mot soninké trouvé dans le lexique.'
for p in (REF1, REF2):
    try:
        audio, sr = sf.read(p)
        assert len(audio) > 0 and sr > 0, f'Audio de référence vide ou invalide : {p}'
    except Exception as exc:
        raise RuntimeError(f'Référence illisible : {p}. Relance la première cellule après avoir placé les audios dans Google Drive. Détail : {exc}') from exc
    print(p.name, '— fréquence:', sr, 'Hz; durée:', round(len(audio)/sr, 2), 's')
print('Mots uniques dans le lexique :', len(words))
print('GPU :', torch.cuda.get_device_name(0))


In [ ]:
# Construire la référence combinée et charger OmniVoice
import sys, subprocess, importlib.util
try:
    from omnivoice import OmniVoice
except ModuleNotFoundError:
    print('Module absent dans ce noyau : tentative d’installation depuis PyPI…')
    p = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--upgrade', 'omnivoice'], capture_output=True, text=True)
    print(p.stdout[-3000:])
    if p.returncode != 0:
        print('Échec installation PyPI :', p.stderr[-5000:])
        print('Tentative depuis le dépôt officiel GitHub…')
        g = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--upgrade', 'git+https://github.com/k2-fsa/OmniVoice.git'], capture_output=True, text=True)
        print(g.stdout[-3000:])
        if g.returncode != 0:
            print(g.stderr[-6000:])
            raise RuntimeError('OmniVoice ne s’installe pas dans ce noyau. Copie ici toute la sortie affichée ci-dessus ; aucun audio n’a été généré.')
    try:
        from omnivoice import OmniVoice
    except Exception as exc:
        print('Python du noyau :', sys.executable)
        print('Module trouvé :', importlib.util.find_spec('omnivoice'))
        raise RuntimeError(f'Installation terminée mais import toujours impossible : {exc}') from exc
a, sr1 = sf.read(REF1); b, sr2 = sf.read(REF2)
if a.ndim > 1: a = a.mean(axis=1)
if b.ndim > 1: b = b.mean(axis=1)
if sr1 != sr2:
    raise ValueError('Les deux audios de référence doivent avoir la même fréquence. Réexporte-les avec la même fréquence puis relance.')
combined_ref = Path('/content/reference_soninke.wav')
sf.write(combined_ref, np.concatenate([a, np.zeros(int(0.35*sr1)), b]), sr1)
model = OmniVoice.from_pretrained('k2-fsa/OmniVoice', device_map='cuda:0', dtype=torch.float16)
print('OmniVoice chargé. Langue demandée : Soninké (snk).')
print('Référence utilisée : Daga goli. Li taxu.')


In [ ]:
# Générer au maximum 20 WAV manquants. Relance cette cellule pour le lot suivant.
if 'model' not in globals():
    raise RuntimeError('ARRÊT : le modèle OmniVoice n’est pas chargé. Exécute d’abord la cellule d’installation, puis la cellule « Construire la référence combinée et charger OmniVoice ». Aucun mot ne sera traité.')
BATCH_SIZE = 20
manifest_path = OUT_DIR / 'manifest_a_valider.csv'
def safe_name(word):
    digest = hashlib.sha256(word.encode('utf-8')).hexdigest()[:16]
    slug = re.sub(r'[^a-zA-Z0-9_-]+', '_', word).strip('_')[:35] or 'mot'
    return f'{slug}_{digest}.wav'

existing = {}
if manifest_path.exists():
    old = pd.read_csv(manifest_path, dtype=str, keep_default_na=False)
    for _, row in old.iterrows():
        existing[row.get('soninke','')] = row.to_dict()

pending = []
for word in words:
    filename = safe_name(word)
    path = OUT_DIR / filename
    row = existing.get(word, {})
    if path.exists() and path.stat().st_size > 44:
        existing[word] = {'soninke': word, 'fichier': filename, 'statut': row.get('statut','À ÉCOUTER — NON VALIDÉ'), 'erreur': ''}
    else:
        pending.append((word, path))

batch = pending[:BATCH_SIZE]
print(f'Mots du lexique : {len(words)} | déjà présents : {len(words)-len(pending)} | à générer : {len(pending)} | ce lot : {len(batch)}')
for i, (word, path) in enumerate(batch, 1):
    if path.exists() and path.stat().st_size <= 44:
        path.unlink()  # supprimer uniquement un fichier manifestement incomplet
    try:
        result = model.generate(text=word, ref_audio=str(combined_ref), ref_text='Daga goli. Li taxu.', language_id='snk')
        samples = np.asarray(result[0]).squeeze()
        if samples.ndim != 1 or samples.size == 0 or not np.isfinite(samples).all():
            raise ValueError('Sortie audio vide ou invalide')
        temp_path = path.with_suffix('.tmp.wav')
        sf.write(temp_path, samples, 24000, subtype='PCM_16')
        check, check_sr = sf.read(temp_path)
        if check.size == 0 or check_sr <= 0:
            temp_path.unlink(missing_ok=True)
            raise ValueError('Vérification WAV échouée')
        if path.exists():
            temp_path.unlink(missing_ok=True)
        else:
            temp_path.replace(path)
        existing[word] = {'soninke': word, 'fichier': path.name, 'statut': 'À ÉCOUTER — NON VALIDÉ', 'erreur': ''}
        print(f'[{i}/{len(batch)}] OK : {word}')
    except Exception as exc:
        existing[word] = {'soninke': word, 'fichier': path.name, 'statut': 'ÉCHEC — À REPRENDRE', 'erreur': str(exc)[:300]}
        print(f'[{i}/{len(batch)}] ÉCHEC : {word} — {exc}')
    pd.DataFrame(list(existing.values())).sort_values('soninke').to_csv(manifest_path, index=False, encoding='utf-8-sig')

pd.DataFrame(list(existing.values())).sort_values('soninke').to_csv(manifest_path, index=False, encoding='utf-8-sig')
print('Lot terminé. Dossier Drive :', OUT_DIR)
print('Manifeste :', manifest_path)
print('Relance cette cellule pour traiter le lot suivant. Aucun audio n’est validé ou publié automatiquement.')


## Écoute et validation humaine

Écoute les WAV dans `Google Drive/SONINKE_TTS_LIBRE/audio_a_valider/`. Le manifeste est volontairement marqué `À ÉCOUTER — NON VALIDÉ`. Ne change ce statut qu'après avoir vérifié la prononciation. Ce notebook ne publie pas les audios dans le site et ne pousse pas de fichiers vers GitHub.

**Limite importante :** Colab n'exécute pas ce notebook en permanence à chaque ajout GitHub. Pour automatiser sans ouvrir Colab, il faut ensuite un runner GPU disponible dans GitHub Actions. Cette version permet de lancer les lots dans Colab et de les sauvegarder durablement dans Drive.